# **03. PostgreSQL Load — загрузка данных в SQL** (вспомогательный ноутбук)

В этом ноутбуке сделаем третий этап проекта **The Movies Dataset** — мы загрузим очищенные данные в PostgreSQL и подготовим SQL-слой для дальнейшего анализа

Цель этого ноутбука — перенести подготовленные данные из CSV-файлов в базу данных PostgreSQL, чтобы дальше работать с ними через SQL

Что сделаем?

- подключим к PostgreSQL через параметры из файла `.env`
- проверим соединение с базой данных
- загрузим таблицы в PostgreSQL
- проверим количество строк после загрузки
- создадим схему `marts` для аналитических витрин
- подготовим базу для создания SQL-витрин

В проекте используются две основные схемы PostgreSQL:

- `clean` — слой очищенных данных после Python-обработки
- `marts` — слой аналитических витрин, подготовленных для анализа и визуализации

---

In [1]:
import pandas as pd
from pathlib import Path
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
from dotenv import load_dotenv
import os

In [2]:
#ищем корень проекта и проверяем, видит ли Python папку processed и файл .env
current_dir = Path.cwd()

if current_dir.name == "notebook":
    project_root = current_dir.parent
else:
    project_root = current_dir

processed_dir = project_root / "data" / "processed"

print("current_dir:", current_dir)
print("project_root:", project_root)
print("processed_dir:", processed_dir)
print("processed_dir exists:", processed_dir.exists())
print(".env exists:", (project_root / ".env").exists())

current_dir: c:\Users\user\Desktop\movie_rating\notebook
project_root: c:\Users\user\Desktop\movie_rating
processed_dir: c:\Users\user\Desktop\movie_rating\data\processed
processed_dir exists: True
.env exists: True


In [3]:
#читаем настройки подключения из .env
load_dotenv(project_root / ".env")

POSTGRES_USER = os.getenv("POSTGRES_USER")
POSTGRES_PASSWORD = os.getenv("POSTGRES_PASSWORD")
POSTGRES_HOST = os.getenv("POSTGRES_HOST")
POSTGRES_PORT = os.getenv("POSTGRES_PORT")
POSTGRES_DB = os.getenv("POSTGRES_DB")

print("Connection settings loaded successfully")
print("database:", POSTGRES_DB)
print("user:", POSTGRES_USER)

Connection settings loaded successfully
database: movies_project
user: postgres


In [4]:
#создаём подключение Python к PostgreSQL
connection_url = URL.create(
    drivername="postgresql+psycopg",
    username=POSTGRES_USER,
    password=POSTGRES_PASSWORD,
    host=POSTGRES_HOST,
    port=POSTGRES_PORT,
    database=POSTGRES_DB
)

engine = create_engine(connection_url)

In [5]:
#проверяем, что Python реально подключился к базе movies_project
with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT 
            current_database() AS database_name,
            current_user AS user_name;
    """))
    
    row = result.fetchone()
    print("database:", row.database_name)
    print("user:", row.user_name)

database: movies_project
user: postgres


In [6]:
#проверяем, какие файлы есть в processed
expected_files = [
    "movies_clean.csv",
    "credits_clean.csv",
    "keywords_clean.csv",
    "ratings_summary.csv",
    "links_clean.csv",
    "movie_base.csv"
]

for file_name in expected_files:
    file_path = processed_dir / file_name
    print(file_name, "->", file_path.exists())

movies_clean.csv -> True
credits_clean.csv -> True
keywords_clean.csv -> True
ratings_summary.csv -> True
links_clean.csv -> True
movie_base.csv -> True


In [7]:
#загружаем CSV в pandas
movies_clean = pd.read_csv(processed_dir / "movies_clean.csv", low_memory=False)
credits_clean = pd.read_csv(processed_dir / "credits_clean.csv", low_memory=False)
keywords_clean = pd.read_csv(processed_dir / "keywords_clean.csv", low_memory=False)
ratings_summary = pd.read_csv(processed_dir / "ratings_summary.csv", low_memory=False)
links_clean = pd.read_csv(processed_dir / "links_clean.csv", low_memory=False)
movie_base = pd.read_csv(processed_dir / "movie_base.csv", low_memory=False)

In [8]:
#проверяем размеры таблиц
tables = {
    "movies_clean": movies_clean,
    "credits_clean": credits_clean,
    "keywords_clean": keywords_clean,
    "ratings_summary": ratings_summary,
    "links_clean": links_clean,
    "movie_base": movie_base
}

for table_name, df in tables.items():
    print(table_name, df.shape)

movies_clean (45433, 15)
credits_clean (45432, 4)
keywords_clean (45432, 3)
ratings_summary (9066, 3)
links_clean (9112, 3)
movie_base (45433, 30)


In [9]:
#создаём схемы ещё раз через Python (уже создали clean и marts в DBeaver, но эта ячейка нужна для подстраховки)
with engine.begin() as conn:
    conn.execute(text("CREATE SCHEMA IF NOT EXISTS clean;"))
    conn.execute(text("CREATE SCHEMA IF NOT EXISTS marts;"))

print("Схемы clean и marts готовы")

Схемы clean и marts готовы


In [10]:
#загружаем таблицы в PostgreSQL
tables_to_upload = {
    "movies_clean": movies_clean,
    "credits_clean": credits_clean,
    "keywords_clean": keywords_clean,
    "ratings_summary": ratings_summary,
    "links_clean": links_clean,
    "movie_base": movie_base
}

for table_name, df in tables_to_upload.items():
    print(f"Загружаю таблицу: {table_name}, размер: {df.shape}")
    
    df.to_sql(
        name=table_name,
        con=engine,
        schema="clean",
        if_exists="replace",
        index=False,
        chunksize=1000,
        method="multi"
    )
    
    print(f"Готово: {table_name}")

print("Все таблицы загружены в PostgreSQL")

Загружаю таблицу: movies_clean, размер: (45433, 15)
Готово: movies_clean
Загружаю таблицу: credits_clean, размер: (45432, 4)
Готово: credits_clean
Загружаю таблицу: keywords_clean, размер: (45432, 3)
Готово: keywords_clean
Загружаю таблицу: ratings_summary, размер: (9066, 3)
Готово: ratings_summary
Загружаю таблицу: links_clean, размер: (9112, 3)
Готово: links_clean
Загружаю таблицу: movie_base, размер: (45433, 30)
Готово: movie_base
Все таблицы загружены в PostgreSQL


In [11]:
#проверяем список таблиц
query = """
SELECT 
    table_schema,
    table_name
FROM information_schema.tables
WHERE table_schema = 'clean'
ORDER BY table_name;
"""

pd.read_sql_query(query, engine)

,table_schema,table_name
0,clean,credits_clean
1,clean,keywords_clean
2,clean,links_clean
3,clean,movie_base
4,clean,movies_clean
5,clean,ratings_summary


In [12]:
#проверяем количество строк в каждой таблице
row_counts_query = """
SELECT 'movies_clean' AS table_name, COUNT(*) AS row_count FROM clean.movies_clean
UNION ALL
SELECT 'credits_clean' AS table_name, COUNT(*) AS row_count FROM clean.credits_clean
UNION ALL
SELECT 'keywords_clean' AS table_name, COUNT(*) AS row_count FROM clean.keywords_clean
UNION ALL
SELECT 'ratings_summary' AS table_name, COUNT(*) AS row_count FROM clean.ratings_summary
UNION ALL
SELECT 'links_clean' AS table_name, COUNT(*) AS row_count FROM clean.links_clean
UNION ALL
SELECT 'movie_base' AS table_name, COUNT(*) AS row_count FROM clean.movie_base
ORDER BY table_name;
"""

pd.read_sql_query(row_counts_query, engine)

,table_name,row_count
0,credits_clean,45432
1,keywords_clean,45432
2,links_clean,9112
3,movie_base,45433
4,movies_clean,45433
5,ratings_summary,9066


In [13]:
#проверяем кусочек movie_base
pd.read_sql_query("""
SELECT 
    id_num,
    title,
    release_year,
    main_genre,
    director,
    avg_rating,
    rating_count,
    weighted_rating,
    high_rating_flag
FROM clean.movie_base
LIMIT 10;
""", engine)

,id_num,title,release_year,main_genre,director,avg_rating,rating_count,weighted_rating,high_rating_flag
0,862,Toy Story,1995.0,Animation,John Lasseter,3.872470,247.0,3.851999,1
1,8844,Jumanji,1995.0,Adventure,Joe Johnston,3.401869,107.0,3.393205,0
2,15602,Grumpier Old Men,1995.0,Romance,Howard Deutch,3.161017,59.0,3.178115,0
3,31357,Waiting to Exhale,1995.0,Comedy,Forest Whitaker,2.384615,13.0,2.755083,0
4,11862,Father of the Bride Part II,1995.0,Comedy,Charles Shyer,3.267857,56.0,3.270951,0
5,949,Heat,1995.0,Action,Michael Mann,3.884615,104.0,3.837273,1
6,11860,Sabrina,1995.0,Comedy,Sydney Pollack,3.283019,53.0,3.284062,0
7,45325,Tom and Huck,1995.0,Action,Peter Hewitt,3.800000,5.0,3.472274,1
8,9091,Sudden Death,1995.0,Action,Peter Hyams,3.150000,20.0,3.193511,0
9,710,GoldenEye,1995.0,Adventure,Martin Campbell,3.450820,122.0,3.439785,0


---

# **вывод по ноутбуку**

В этом ноутбуке очищенные данные из папки `data/processed` были загружены в PostgreSQL. Для хранения подготовленных таблиц использовалась схема `clean`, куда были добавлены таблицы `movies_clean`, `credits_clean`, `keywords_clean`, `ratings_summary`, `links_clean` и `movie_base`

После загрузки была выполнена проверка количества строк, чтобы убедиться, что данные перенесены в базу корректно. Также была создана схема `marts` для аналитических витрин. Сами витрины формируются отдельным SQL-скриптом `sql/create_marts_postgres.sql`, который использует таблицу `clean.movie_base` как основной источник данных

Этот ноутбук выполняет техническую связку между этапом очистки данных и аналитическим этапом. После загрузки данных в PostgreSQL можно выполнить SQL-скрипт создания витрин и использовать готовые таблицы в следующем ноутбуке `04_analysis.ipynb`.